# Hip 라벨 댓글 크롤링 v2 (URL 직접 지정)

**목적:** `purpose = train` — RoBERTa 학습용  
**베이스:** 수빈 코드 구조 + Colab Secrets API 키 + 한글 ≥ 20% 필터  
**저장 경로:** Google Drive `/FNC/data/train/hip/`

### 실행 전 필요한 것
1. **YouTube Data API v3 키** — Colab 왼쪽 사이드바 🔑(Secrets) → `YOUTUBE_API_KEY` 이름으로 등록
2. **VIDEOS 셀에 URL 입력** — `video_type`은 수기로 작성 (`MV` / `teaser` / `dance` / `fancam` / `behind` / `others`) others의 경우 주로 performance video 등

In [ ]:
# Cell 0 — Colab 환경 설정
from google.colab import drive
drive.mount('/content/drive')
!pip install -q google-api-python-client

In [ ]:
# Cell 1 — 설정
import json
import re
import hashlib
import html as html_module
from datetime import datetime, timezone
from urllib.parse import urlparse, parse_qs
from pathlib import Path

import pandas as pd
from googleapiclient.discovery import build
from google.colab import userdata

# ── API 키 (Colab Secrets에서 불러오기) ──────────────────────
API_KEY = userdata.get("YOUTUBE_API_KEY")
# ────────────────────────────────────────────────────────────

LABEL        = "hip"
MAX_COMMENTS = 999999
SAVE_DIR     = Path("/content/drive/MyDrive/FNC/data/train/hip")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

youtube = build("youtube", "v3", developerKey=API_KEY)
print("설정 완료")

In [ ]:
# Cell 2 — 영상 목록 (수기 입력)
# ── URL 부분에 video_type은 수기로 작성 ──────────────────────
VIDEOS = [
    #Cortis
    {"url": "https://youtu.be/qSYNCpb0Bzw?si=YpEmFMc8L0pYjKlK", "artist": "Cortis",    "video_type": "dance",        "purpose": "train"},
    {"url": "https://youtu.be/-fTvZfQoF-w?si=6pgeF6dhzmRO3OXD", "artist": "Cortis",    "video_type": "others",       "purpose": "train"},
    {"url": "https://youtu.be/U6BDbXIah-Y?si=Uu3Isq2nN51P_SYM", "artist": "Cortis",    "video_type": "MV",           "purpose": "train"},
    {"url": "https://youtu.be/42wfEs7oIP8?si=KJTz6oSQndQWGPvo", "artist": "Cortis",    "video_type": "MV",           "purpose": "train"},
    {"url": "https://youtu.be/e2OpbOv_JiQ?si=h-VCEfoYl7R2CqZm", "artist": "Cortis",    "video_type": "MV",           "purpose": "train"},
    {"url": "https://youtu.be/u5IhtyOsEzE?si=wI5llH1_80x_rya5", "artist": "Cortis",    "video_type": "dance",        "purpose": "train"},
    #ALLDAYPROJECT
    {"url": "https://youtu.be/VjvzYjU1mY0?si=PrgFT0m4diSmh1cJ", "artist": "ALLDAYPROJECT", "video_type": "MV",       "purpose": "train"},
    {"url": "https://youtu.be/mhKCRnUKp5U?si=hNYwvcJ8f70kcedA", "artist": "ALLDAYPROJECT", "video_type": "others",   "purpose": "train"},
    {"url": "https://youtu.be/OgEwJ8a1OoY?si=5BoiR1mvjM4UfWFX", "artist": "ALLDAYPROJECT", "video_type": "MV",       "purpose": "train"},
    {"url": "https://youtu.be/9nOrPNxjvck?si=Jmx8E8-0--kDVOxe", "artist": "ALLDAYPROJECT", "video_type": "MV",       "purpose": "train"},
    # Stray_Kids
    {"url": "https://youtu.be/TQTlCHxyuu8?si=2WQwPyIsBkE4zX0j", "artist": "Stray_Kids", "video_type": "MV",          "purpose": "train"},
    {"url": "https://youtu.be/ovHoY8UBIu8?si=fkZ4zvvchCm2zkmG", "artist": "Stray_Kids", "video_type": "MV",          "purpose": "train"},
    {"url": "https://youtu.be/Koi6AZyT_HU?si=oMu8uccWHdowb5xt", "artist": "Stray_Kids", "video_type": "dance",       "purpose": "train"},
    # NCT127
    {"url": "https://youtu.be/2OvyA2__Eas?si=kzL_mVgDraRpHljA", "artist": "NCT127",   "video_type": "MV",            "purpose": "train"},
    {"url": "https://youtu.be/WkuHLzMMTZM?si=6umeo6-aSiOzB3OK", "artist": "NCT127",   "video_type": "MV",            "purpose": "train"},
    # BTS
    {"url": "https://youtu.be/kTlv5_Bs8aw?si=rZZ44M2PzGS4bffj", "artist": "BTS",   "video_type": "MV",               "purpose": "train"},
    {"url": "https://youtu.be/rBG5L7UsUxA?si=0AypwMuOSSNVmY1l", "artist": "BTS",   "video_type": "MV",               "purpose": "train"},
    # NCT_U
    {"url": "https://youtu.be/gvXsmI3Gdq8?si=jRJi_j5y_gb8_wzW", "artist": "NCT_U",   "video_type": "MV",               "purpose": "train"},
    {"url": "https://youtu.be/0b-EatqUvb4?si=7sBQ0udovEd28Hw3", "artist": "NCT_U",   "video_type": "others",           "purpose": "train"},
    {"url": "https://youtu.be/sGhuwRYp5hM?si=g23DkI_9iJb9BCFE", "artist": "NCT_U",   "video_type": "dance",            "purpose": "train"}
]
# ────────────────────────────────────────────────────────────
print(f"수집 대상: {len(VIDEOS)}개 영상")

In [ ]:
# Cell 3 — 헬퍼 함수

# ── video_id 추출 (수빈 코드 그대로) ─────────────────────────
def extract_video_id(url):
    url = url.strip()
    match = re.match(r"(?:https?://)?youtu\.be/([A-Za-z0-9_-]{11})", url)
    if match:
        return match.group(1)
    match = re.match(r"(?:https?://)?(?:www\.)?youtube\.com/shorts/([A-Za-z0-9_-]{11})", url)
    if match:
        return match.group(1)
    qs = parse_qs(urlparse(url).query)
    if "v" in qs:
        return qs["v"][0]
    return None

# ── 한글 비율 필터 (hip_crawler.ipynb에서 가져옴) ─────────────
HANGUL_RE   = re.compile(r"[가-힣]")
HTML_TAG_RE = re.compile(r"<[^>]+>")
SPAM_RE     = re.compile(r"http[s]?://|www\.|\.com|\.kr")
REPEAT_RE   = re.compile(r"(.)\1{4,}")

def has_enough_korean(text: str, min_ratio: float = 0.2) -> bool:
    if not text:
        return False
    return len(HANGUL_RE.findall(text)) / len(text) >= min_ratio

def clean(records: list) -> list:
    seen, cleaned = set(), []
    for r in records:
        text = HTML_TAG_RE.sub(" ", r["text"].strip())
        text = html_module.unescape(text)
        text = re.sub(r" {2,}", " ", text).strip()
        if len(text) < 4:
            continue
        h = hashlib.md5(text.encode()).hexdigest()
        if h in seen:
            continue
        seen.add(h)
        if SPAM_RE.search(text) or REPEAT_RE.search(text):
            continue
        if not has_enough_korean(text):
            continue
        r["text"] = text
        cleaned.append(r)
    return cleaned

print("헬퍼 함수 로드 완료")

In [ ]:
# Cell 4 — 댓글 수집 함수 (수빈 코드 구조 그대로)
def crawl_video(video_cfg):
    video_id = extract_video_id(video_cfg["url"])
    if not video_id:
        print(f"[ERROR] 유효하지 않은 링크: {video_cfg['url']}")
        return []

    info = youtube.videos().list(part="snippet", id=video_id).execute()
    if not info["items"]:
        print(f"[ERROR] 영상 없음 (삭제/비공개): {video_cfg['url']}")
        return []
    snippet            = info["items"][0]["snippet"]
    video_title        = snippet["title"]
    video_published_at = snippet["publishedAt"]

    print(f"[{video_id}] 크롤링 시작 - {video_title}")

    records         = []
    next_page_token = None
    crawled_at      = datetime.now(timezone.utc).isoformat()

    while True:
        try:
            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(MAX_COMMENTS - len(records), 100),
                pageToken=next_page_token,
                textFormat="plainText",
            ).execute()
        except Exception as e:
            print(f"  [{video_id}] 오류: {e}")
            break

        for item in response["items"]:
            comment = item["snippet"]["topLevelComment"]["snippet"]
            records.append({
                "video_id":           video_id,
                "video_type":         video_cfg["video_type"],
                "video_title":        video_title,
                "video_published_at": video_published_at,
                "artist":             video_cfg["artist"],
                "label":              LABEL,
                "purpose":            video_cfg["purpose"],
                "comment_id":         item["snippet"]["topLevelComment"]["id"],
                "text":               comment["textDisplay"],
                "likes":              comment["likeCount"],
                "published_at":       comment["publishedAt"],
                "crawled_at":         crawled_at,
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token or len(records) >= MAX_COMMENTS:
            break

    print(f"[{video_id}] 완료 - {len(records)}개 댓글 수집")
    return records

print("수집 함수 로드 완료")

In [ ]:
# Cell 5 — 전체 실행
all_records = []
START_FROM = 11

for video_cfg in VIDEOS[START_FROM:]:
    raw     = crawl_video(video_cfg)
    cleaned = clean(raw)
    all_records.extend(cleaned)
    print(f"  → 전처리 후: {len(cleaned)}개  (원본 {len(raw)}개)")

print(f"\n전체: {len(all_records)}개")

In [ ]:
# Cell 6 — Google Drive 저장
if not all_records:
    print("⚠️   수집된 데이터 없음")
else:
    df_new = pd.DataFrame(all_records)

    for artist, group in df_new.groupby("artist"):
        fname = SAVE_DIR / f"{artist}.jsonl"
        if fname.exists():
            df_existing = pd.read_json(fname, orient="records", lines=True)
            group = pd.concat([df_existing, group], ignore_index=True)
        group.to_json(fname, orient="records", lines=True, force_ascii=False)
        print(f"저장 완료: {fname}  ({len(group):,}개)")

    all_fname = SAVE_DIR / "hip_all.jsonl"
    if all_fname.exists():
        df_existing_all = pd.read_json(all_fname, orient="records", lines=True)
        df_new = pd.concat([df_existing_all, df_new], ignore_index=True)
    df_new.to_json(all_fname, orient="records", lines=True, force_ascii=False)
    print(f"저장 완료: {all_fname}  (총 {len(df_new):,}개)")

In [ ]:
# Cell 7 — 검증
if not all_records:
    print("데이터 없음")
else:
    print("아티스트별")
    print(df.groupby("artist")["text"].count().to_string())

    print("\n영상 타입별")
    print(df.groupby("video_type")["text"].count().to_string())

    print("\npublished_at 형식 확인 (ISO 8601 UTC)")
    for s in df["published_at"].head(3):
        assert "T" in s and "Z" in s, f"형식 오류: {s}"
        print(f"  OK: {s}")

    print("\n샘플 댓글 5개")
    for t in df["text"].sample(min(5, len(df)), random_state=42):
        print(f"  • {t}")